In [7]:
"""
02 - Autoencoder on a 8x8 handwritten digits. 
"""
# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Scikit learn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [8]:
# Setting the PyTorch seed
SEED = 373
torch.manual_seed(SEED)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
# 1. Data
# Loading the dataset from load_digts | 1797 images
digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = digits.target                                           # y in autoencoder is not part of the loss

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
train_dl = DataLoader(TensorDataset(X_tr), batch_size=64, shuffle=True)
val_dl = DataLoader(TensorDataset(X_va), batch_size=512)

In [31]:
X

tensor([[0.0000, 0.0000, 0.3125,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.6250, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 1.0000, 0.5625, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0625,  ..., 0.3750, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1250,  ..., 0.7500, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6250,  ..., 0.7500, 0.0625, 0.0000]])

In [ ]:
# 2. Models

## 2.1 Autoencoder
class AutoEncoder(nn.Module):
    def __init__(self, d_in=64, d_hidden=128, d_latent=8):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_in), nn.Sigmoid(), # output in [0, 1] like the input
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def loss(self, x):
        x_hat, _ = self(x)
        return F.mse_loss(x_hat, x)


## 2.2 VAE
class VAE(Autoencoder):
    """Same shape, but encoder outputs (mu, logvar) and we add a KL term."""
    def __init__(self, d_in=64, d_hidden=128, d_latent=8, beta=1.0):
        super().__init__(d_in, d_hidden, d_latent)
        self.encoder[-1] = nn.Linear(d_hidden, 2 * d_latent) # -> mu, logvar
        self.beta = beta

    def forward(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=-1)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu) if self.training else mu
        return self.decoder(z), (mu, logvar)

    def loss(self, x):
        x_hat, (mu, logvar) = self(x)
        recon = F.binary_cross_entropy(x_hat, x, reduction="sum") / len(x)
        kl = -0.5 * torch.sum(mu + logvar - mu.pow(2) - logvar.exp()) / len(x)
        return recon + self.beta * kl

    @torch.inference_mode()
    def sample(self, n):
        z = torch.randn(n, self.decoder[0].in_features, device=next(self.parameters()).device)
        return self.decoder(z)
    


array([0, 1, 2, ..., 8, 9, 8], shape=(1797,))

In [ ]:
# 3. Generic train loop
# print(f"  epoch {epoch:3d} | train loss {loss.item():.4f} | val loss {va:.4f} | val MSE {mse:.4f}")
def fit(model, epochs=100, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for epoch in range(1, epochs+1):
        model.train()
        for (xb,) in train_dl:
            xb = xb.to(device)
            loss = model.loss(xb)
            optimizer.zero_grad(set_to_none=True)
            loss.backtward()
            optimizer.step()

        if epoch % 25 == 0:
            model.eval()
            with torch.inference_mode():
                va = sum(model.loss(xb.to(device)).item() * len(xb) for (xb,) in val_dl) / len(X_va)
                mse = F.mse_loss(model(X_va.to(device))[0], X_va.to(device)).item()
            print(f"    epoch {epoch:3d} | train loss {loss.item():.4f} | val loss {va:.4f} | val MSE {mse:.4f}")
    
    return model

@torch.inference_mode
def encode(model, X):
    model.eval()
    out = model(X.to(device))[1]
    return (out[0] if isinstance(out, tuple) else out).cpu().numpy() # use mu for VAE

In [45]:
a = [1, 2, 3]
a[-1] = 4
print(a)

[1, 2, 4]


In [ ]:
torch.randn_like()

NotImplementedError: "normal_kernel_cpu" not implemented for 'Long'

In [17]:
digits.data[0]

array([ 0.,  0.,  5., 13.,  9.,  1.,  0.,  0.,  0.,  0., 13., 15., 10.,
       15.,  5.,  0.,  0.,  3., 15.,  2.,  0., 11.,  8.,  0.,  0.,  4.,
       12.,  0.,  0.,  8.,  8.,  0.,  0.,  5.,  8.,  0.,  0.,  9.,  8.,
        0.,  0.,  4., 11.,  0.,  1., 12.,  7.,  0.,  0.,  2., 14.,  5.,
       10., 12.,  0.,  0.,  0.,  0.,  6., 13., 10.,  0.,  0.,  0.])